In [27]:
import pickle
from torch import nn
import os
import torch
import shap

In [ ]:
def load_weights(read_dir, model_type):
    """
    Loads weights and biases from the specified directory.

    Args:
        read_dir (str): Path to the directory containing the weight files.
        model_type (str): 'decoder' or 'classifier'.
    
    Returns:
        weights_list, biases_list (list): Loaded weights and biases.
    """
    with open(os.path.join(read_dir, f'{model_type}_weights_list.pkl'), 'rb') as f:
        weights_list = pickle.load(f)

    with open(os.path.join(read_dir, f'{model_type}_biases_list.pkl'), 'rb') as f:
        biases_list = pickle.load(f)

    return weights_list, biases_list

# A wrapper over tenosorflow weights to load in pytorch and apply SHAP
class AdjustedModel(nn.Module):
    def __init__(self, weights_list, biases_list, layer_dims):
        """
        Args:
            weights_list (list): List of weights for each layer.
            biases_list (list): List of biases for each layer.
            layer_dims (list): List containing layer dimensions for nn.Linear.
        """
        super(AdjustedModel, self).__init__()
        
        layers = []
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))
            if i < len(layer_dims) - 2:
                layers.append(nn.ReLU())

        self.features = nn.Sequential(*layers)
        
        # Initialize weights and biases
        with torch.no_grad():
            print("Verifying model loading")
            layer_idx = 0
            for layer in self.features:
                if isinstance(layer, nn.Linear):
                    print(f"Layer {layer_idx}: {layer}")
                    print("Weights shape:", weights_list[layer_idx].shape, 
                          "Biases shape:", biases_list[layer_idx].shape)

                    layer.weight = nn.Parameter(torch.Tensor(weights_list[layer_idx]).T)
                    layer.bias = nn.Parameter(torch.Tensor(biases_list[layer_idx]))
                    layer_idx += 1

    def forward(self, x):
        return self.features(x)



In [29]:
chosen_dataset='Pancreas'
chosen_experiment='results'

In [33]:
read_dir = os.path.join('..', chosen_dataset, chosen_experiment, 'chosen_exp_components')

# Load decoder and classifier weights
decoder_weights_list, decoder_biases_list = load_weights(read_dir, 'decoder')
classifier_weights_list, classifier_biases_list = load_weights(read_dir, 'classifier')

# Define layer dimensions for both models
decoder_dims = [6, 64, 512, 2000]  # Example layer dimensions
classifier_dims = [6,8,7]        # Example layer dimensions

# Initialize models
decoder_model = AdjustedModel(decoder_weights_list, decoder_biases_list, decoder_dims)
classifier_model = AdjustedModel(classifier_weights_list, classifier_biases_list, classifier_dims)

# --- Load Additional Files ---
with open(os.path.join(read_dir, 'z_values.pkl'), 'rb') as f:
    z_values = pickle.load(f)

with open(os.path.join(read_dir,'index_to_class.pkl'), 'rb') as f:
    index_to_class = pickle.load(f)



Verifying model loading
Layer 0: Linear(in_features=6, out_features=64, bias=True)
Weights shape: (6, 64) Biases shape: (64,)
Layer 1: Linear(in_features=64, out_features=512, bias=True)
Weights shape: (64, 512) Biases shape: (512,)
Layer 2: Linear(in_features=512, out_features=2000, bias=True)
Weights shape: (512, 2000) Biases shape: (2000,)
Verifying model loading
Layer 0: Linear(in_features=6, out_features=8, bias=True)
Weights shape: (6, 8) Biases shape: (8,)
Layer 1: Linear(in_features=8, out_features=7, bias=True)
Weights shape: (8, 7) Biases shape: (7,)


In [ ]:
for model_type, model in {'decoder': decoder_model, 'classifier': classifier_model}.items():
    print(f"Processing {model_type}")
    
    # Apply interleaving only for the decoder
    interleave_frequency = 10 if model_type == "decoder" else 1
    
    # Define SHAP explainer with interleaving or full data
    explainer = shap.DeepExplainer(model, torch.Tensor(z_values)[::interleave_frequency, :])
    
    shap_values = {}
    for i in range(7):
        shap_values[i] = explainer.shap_values(torch.Tensor(z_values[index_to_class[i], :]))
        print(f"{model_type} - Cell type {i} completed")
    
    # Save the SHAP values
    output_path = os.path.join(read_dir, f'SHAP_Values_DeepExplainer_{interleave_frequency}_{model_type}.pkl')
    with open(output_path, 'wb') as f:
        pickle.dump(shap_values, f)

    print(f"Saved SHAP values for {model_type} at {output_path}")
